## Import packages

In [ ]:
import os, sys
import pandas as pd
import numpy as np
from pym3c.hic import mcds2frac
from scipy.stats import zscore, pearsonr, norm
import cooler
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pylab as plt
import matplotlib as mpl
import seaborn as sns
mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Arial'
mpl.rcParams['figure.dpi'] = 80
mpl.rcParams['savefig.dpi']=300
from PyComplexHeatmap import *

## Parameters

In [ ]:
workdir = "/home/x-wding2/project_wubin/BG/HIC/BICAN_pseudobulk"
chrom_size_path = "/home/x-wding2/Ref/hg38/hg38_ucsc.nochrM.sizes"
metadata_path = "/home/x-wding2/Projects/BG/clustering/100kb/cell_metadata_with_annotation.tsv"
pseudobulk_mcds_path = "/home/x-wding2/project_wubin/BG/pseudobulk/Leaf/Study+Name+donor+Region+Subclass.5kb.mcds" #"5kb", used to calculate correlation between mC and compartment score
pseudobulk_atac_adata_path=None
palette_path = "/home/x-wding2/Projects/BG/metadata/HMBA_color_palette.xlsx"
figdir=None
mode='raw'
res=100000
var_dim5kb="5kb"
group_col = "Subclass"
parent_col = "Class" #used to sort the groupby (MajorType)
sort_cols=None
stratify_col = None
stratify_figdir=None
split_order = ["Nonneuron", "Glut", "Splatter", "Subpallium GABA"]
diff_comp_dir=None
extract_group=None
query_group=None
normalize_per_cell=True
kernel_name = "m3c"
excluded_groups=['Low quality cells','doublet','Unk']

extract_group = [4]
query_group = {"0": "BICAN"}

extract_group = [4,3]
group_col = "Region"
stratify_col = "Subclass"

In [ ]:
# Parameters
workdir = "/home/x-wding2/Projects/BG/HIC/BG_Class"
metadata_path = "/home/x-wding2/Projects/BG/clustering/100kb/annotations_without_Glut.tsv"
adata_prefix = "/home/x-wding2/Projects/BG/adata/BG"
atac_adata_path = "/home/x-wding2/Projects/BG/atac/HMBA.atac-5kb.h5ad"
atac_ann_path = "/home/x-wding2/Projects/BG/atac/obs.tsv"
pseudobulk_mcds_path = "/home/x-wding2/Projects/BG/pseudobulk/Class/Class.mcds"
pseudobulk_atac_adata_path = "/home/x-wding2/Projects/BG/atac/Pseudobulk.Class.5kb.h5ad"
chrom_size_path = "/home/x-wding2/Ref/hg38/hg38_ucsc.nochrM.sizes"
palette_path = "/home/x-wding2/Projects/BG/metadata/BG_color_palette.xlsx"
cg_count_path = "/home/x-wding2/Ref/hg38/annotations/CG-count-100k.tsv"
gene_annotation = "/home/x-wding2/Ref/hg38/annotations/gene_2000_2000.bed.gz"
marker_path = "/home/x-wding2/Projects/BG/metadata/BG_markers.xlsx"
marker_sheet = "BG"
mode = "raw"
prefix = "Class"
group_col = "Class"
parent_col = "Neighborhood"
sort_cols = ["Neighborhood"]
stratify_col = None
loop_prefix = None
query_str = None
excluded_groups = ["Low quality cells", "doublet", "Unk", "Glut"]
figdir = None
cpu = 8
indir = None
kernel_name = "m3c"
genome = "hg38"
obs_dim = "cell"
var_dim = "100kb"
var_dim5kb = "5kb"
extract_group = None
query_group = None
mc_types = ["CGN", "CHN"]
normalize_per_cell = True
split_order = ["Nonneuron", "Subpallium GABA", "Glut Sero Dopa", "Subpallium GABA-Glut"]
diff_comp_dir = None
dchic_env = "dchic"
param_yaml = "/home/x-wding2/Projects/Github/pym3c/pym3c/notebooks/templates/BG_Subclass_hic.yaml"
outdir = "/home/x-wding2/Projects/BG/HIC/BG_Class/compartment"
outname = "4.Class.plot_diff_compartment.ipynb"


In [ ]:
if not workdir is None:
    workdir=os.path.expanduser(workdir)
if figdir is None:
    assert not workdir is None
    figdir=os.path.join(workdir,"compartment",f"{mode}_{group_col}")
else:
    figdir=os.path.expanduser(figdir)
if not os.path.exists(figdir):
    os.makedirs(figdir, exist_ok=True)
chrom_size_path=os.path.expanduser(chrom_size_path)
metadata_path=os.path.expanduser(metadata_path)
chrom_size_path=os.path.expanduser(chrom_size_path)

if diff_comp_dir is None:
    diff_comp_dir=os.path.join(workdir,"compartment",f"{group_col}DiffComp")
diff_comp_dir=os.path.expanduser(diff_comp_dir)
print(figdir,diff_comp_dir,group_col)

In [ ]:
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes[chrom_sizes.index!='chrY']
df_chr=chrom_sizes.to_frame()
df_chr['N']=df_chr.index.to_series().apply(lambda x:int(x.lstrip('chr')) if x.lstrip('chr') not in ['X','Y'] else 30) #sort by chrom int (1,2,3..)
df_chr.sort_values('N',inplace=True)
chrom_sizes=chrom_sizes.loc[df_chr.index.tolist()]
if metadata_path.endswith('.csv') or metadata_path.endswith('.csv.gz'):
    metadata = pd.read_csv(metadata_path, index_col=0)
else:
    metadata = pd.read_csv(metadata_path, index_col=0,sep='\t')
metadata=metadata.loc[~ (metadata[group_col].isin(excluded_groups))]
metadata=metadata.loc[metadata[group_col].apply(lambda x:not str(x).endswith('-Unk'))]
if not stratify_col is None:
    metadata=metadata.loc[~ (metadata[stratify_col].isin(excluded_groups))]
if not parent_col is None:
    metadata=metadata.loc[~ (metadata[parent_col].isin(excluded_groups))]
if sort_cols is None:
    sort_cols=parent_col
print(metadata.shape)
metadata.head()

## Define group_order and colors

In [ ]:
# read transformed X_pca for each cell type generated by 1.call_compartment.ipynb
comp = pd.read_csv(os.path.join(figdir,"compartment_score_pcs.tsv.gz"),sep='\t',index_col=0)
print(comp.shape)
comp.head()

In [ ]:
# define group_order (such as cell types) and stratify2group (such as Subclass to Subclass+Region)
if not sort_cols is None:
    metadata.sort_values(sort_cols,inplace=True)
else:
    metadata.sort_values(group_col,inplace=True)

group_order=comp.columns.tolist()
if not parent_col is None:
    parent2group=metadata.loc[:,[parent_col,group_col]].drop_duplicates().groupby(parent_col)[group_col].apply(lambda x:x.unique().tolist()).to_dict()
else:
    parent2group={}
if not stratify_col is None:
    if stratify_figdir is None:
        metadata['GroupID']=metadata[stratify_col].map(str)+'+'+metadata[group_col].map(str) #apply(lambda x:x.replace(' ','_')).
    else:
        metadata['GroupID']=metadata[group_col]
    stratify2group=metadata.loc[:,[stratify_col,'GroupID']].drop_duplicates().groupby(stratify_col)['GroupID'].apply(lambda x:x.unique().tolist()).to_dict()
    # group2stratify=metadata.loc[:,[stratify_col,'GroupID']].drop_duplicates().set_index('GroupID')[stratify_col].to_dict()
    if not parent_col is None:
        group2parent=metadata.loc[:,[parent_col,'GroupID']].drop_duplicates().set_index('GroupID')[parent_col].to_dict()
    else:
        group2parent={} # group is either cell types (such ODC) or GroupID (such as ODC+Pu), parent is Class (such as Glut, Nonneuron)
else:
    stratify2group={}
    # group2stratify={}
    if not parent_col is None:
        group2parent=metadata.loc[:,[parent_col,group_col]].drop_duplicates().set_index(group_col)[parent_col].to_dict()
    else:
        group2parent={}
group2parent['merged']='merged'
parent2group['merged']=metadata[group_col].unique().tolist()
print(group_order)
print(stratify2group)
# print(group2stratify)
print(group2parent)
print(parent2group)

In [ ]:
# read color palette
D=pd.read_excel(os.path.expanduser(palette_path),
                sheet_name=None, index_col=0)
group_colors=D[group_col].Hex.to_dict()
if not parent_col is None:
    parent_colors=D[parent_col].Hex.to_dict()
    parent_colors['merged']='black'
else:
    parent_colors=None
if not stratify_col is None:
    D1=D[stratify_col].Hex.to_dict()
    stratify_colors={k: D1[k] for k in D1 if k in stratify2group}
else:
    stratify_colors={}
print(group_colors)
print(parent_colors)
print(stratify_colors)

## Plot for parent / stratify

In [ ]:
frac_dict=mcds2frac(pseudobulk_mcds_path=pseudobulk_mcds_path,
					   var_dim=var_dim5kb,res=res,mc_types=['CGN','CHN'],
					   extract_group=extract_group,query_group=query_group,
                    filter_chroms=True,normalize_per_cell=normalize_per_cell)

In [ ]:
from pym3c.adata import merge_adata_regions
if not pseudobulk_atac_adata_path is None:
    atac_data=merge_adata_regions(pseudobulk_atac_adata_path,bin_size=5000,use_raw=True,
    		res=res,filter_chroms=True)
    atac_data=atac_data.T
    print(atac_data.shape)
    # print(atac_data.head())
else:
    atac_data=None

In [ ]:
use_dict=parent2group if stratify_col is None else stratify2group
for parent in use_dict:
    parent_name=parent.replace(' ','_')
    infile=os.path.join(
        diff_comp_dir,
        f'{parent_name}/DifferentialResult/{parent_name}/fdr_result/differential.intra_sample_combined.pcQnm.bedGraph')
    if not os.path.exists(infile):
        print(f"Skip {parent_name}: {infile} not existed")
        continue
    if not os.path.exists(os.path.join(figdir,f'{parent_name}_compartment_correlation.txt.gz')):
        # read quantile-normalized compartment results (dchic)
        df_comp = pd.read_csv(infile,sep='\t')
        df_comp.index = df_comp['chr'] + '_' + (df_comp['start'] // res).astype(str)
        df_comp.columns=[col.replace('_AND_','-').replace('_',' ').replace('_PLUS_','+') if col!="sample_maha" else col for col in df_comp.columns.tolist()]
        df_comp=df_comp.loc[~ df_comp['chr'].isin(['chrX','chrM','chrY'])]
        df_bin = df_comp[['chr', 'start', 'end', 'sample_maha', 'pval', 'padj']]
        usecols=list(set(use_dict[parent]) & set(df_comp.columns.tolist()))
        comp = df_comp[usecols]

        mcg=frac_dict['CGN'] # methylation fraction for each 100kb bin in each cell type
        use_cols=list(set(mcg.columns.tolist()) & set(usecols))
        mch=frac_dict['CHN']
        comp=comp.loc[:,use_cols]

        df_bin['mCG_corr'] = [pearsonr(xx, yy)[0] for xx,yy in zip(comp.values, mcg.loc[df_bin.index.tolist(),use_cols].values)] #for each bin
        df_bin['mCH_corr'] = [pearsonr(xx, yy)[0] for xx,yy in zip(comp.values, mch.loc[df_bin.index.tolist(),use_cols].values)]
        df_bin['logPadj'] = df_bin['padj'].apply(lambda x:-np.log10(x))
        df_bin.loc[df_bin['logPadj']>300, 'logPadj'] = 300
        df_bin['mCG_std'] = np.std(mcg, axis=1) #rows
        df_bin['mCH_std'] = np.std(mch, axis=1)
        df_bin['comp_std'] = np.std(comp, axis=1)
        if not pseudobulk_atac_adata_path is None:
            keep_cols=list(set(use_cols) & set(atac_data.columns.tolist()))
            if len(keep_cols) < 2:
                print(keep_cols)
                continue
            df_bin['ATAC_corr'] = [pearsonr(xx, yy)[0] for xx,yy in zip(comp.loc[df_bin.index.tolist(),keep_cols].values, 
                                                                        atac_data.loc[df_bin.index.tolist(),keep_cols].values)] #for each bin
            df_bin['ATAC_std'] = np.std(atac_data.loc[df_bin.index.tolist(),keep_cols].values, axis=1) #rows
        df_bin.to_csv(os.path.join(figdir,f'{parent_name}_compartment_correlation.txt.gz'),sep='\t')
    else:
        df_bin=pd.read_csv(os.path.join(figdir,f'{parent_name}_compartment_correlation.txt.gz'),sep='\t',index_col=0)
# sample_maha: This is the Mahalanobis distance for a specific sample (or bin), calculated from the compartment scores (usually PC1) across samples or groups.
# It measures how far a point (sample) is from the mean distribution of the reference group, taking into account the covariance structure of the data.
# Higher values mean that the region’s compartment score is more different from the average of the control/other group.

## Plot for merged cell types (all cell types together) or each stratify

In [ ]:
def df2bw(data,chrom_size_path,outfile,col,chrom="chrom",start='start',end='end'):
    import cooler,pyBigWig
    chrom_size_path = os.path.expanduser(chrom_size_path)
    chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
    chrom_sizes = chrom_sizes[chrom_sizes.index != 'chrY']
    df_chr = chrom_sizes.to_frame()
    df_chr['N'] = df_chr.index.to_series().apply(
        lambda x: int(x.lstrip('chr')) if x.lstrip('chr') not in ['X', 'Y'] else 30)  # sort by chrom int (1,2,3..)
    df_chr.sort_values('N', inplace=True)
    chrom_sizes = chrom_sizes.loc[df_chr.index.tolist()]
    chrom_sizes_list = [(k, v) for k, v in chrom_sizes.to_dict().items()]

    data.sort_values([chrom,start,end],ascending=True,inplace=True)
    f = pyBigWig.open(outfile, "w")
    f.addHeader(chrom_sizes_list)
    for c in chrom_sizes.index: #order of chrom should be the same as chrom_sizes_list
        tmp1=data.loc[data[chrom] == c]
        if tmp1.shape[0]==0:
            continue
        f.addEntries(tmp1[chrom].tolist(), tmp1[start].tolist(),ends=tmp1[end].tolist(),values=tmp1[col].tolist(),validate=True)
    f.close()

def get_comp_data(group,use_dict,folder_name=None):
    group_name=group.replace(' ','_')
    if folder_name is None:
        infile=os.path.join(diff_comp_dir,f'{group_name}/DifferentialResult/{group_name}/fdr_result/differential.intra_sample_combined.pcQnm.bedGraph')
    else:
        folder_name=folder_name.replace(' ','_')
        infile=os.path.join(diff_comp_dir,f'{folder_name}/DifferentialResult/{folder_name}/fdr_result/differential.intra_sample_combined.pcQnm.bedGraph')
    if not os.path.exists(infile):
        print(infile)
        return False
    df_comp = pd.read_csv(infile, sep='\t')
    df_comp.index = df_comp['chr'] + '_' + (df_comp['start'] // res).astype(str)
    df_comp.columns=[col.replace('_AND_','-').replace('_',' ').replace('_PLUS_','+') for col in df_comp.columns.tolist()]
    df_comp=df_comp.loc[~ df_comp['chr'].isin(['chrX','chrM','chrY'])]
    # df_bin = df_comp[['chr', 'start', 'end', 'sample_maha', 'pval', 'padj']]
    usecols=list(set(use_dict[group]) & set(df_comp.columns.tolist()))
    comp = df_comp[usecols]
    # write normalized compartment scores into bigwig files
    outdir=os.path.join(figdir,"compartment_scores")
    if not os.path.exists(outdir):
        os.mkdir(outdir)
    comp1=df_comp.loc[:,['chr','start','end']+usecols]
    for col in comp1.columns.tolist()[3:]:
        print(col,end=',')
        outfile=os.path.join(outdir,f"{col.replace(' ','_')}.bw")
        df2bw(comp1,chrom_size_path,outfile,col,chrom="chr",start='start',end='end')
        outfile=os.path.join(outdir,f"{col.replace(' ','_')}.bed")
        comp1.loc[:,['chr','start','end',col]].to_csv(outfile,sep='\t',index=False,header=False)

    if not os.path.exists(os.path.join(figdir,f'{group_name}_compartment_correlation.txt.gz')):
        return False
    df_bin=pd.read_csv(os.path.join(figdir,f'{group_name}_compartment_correlation.txt.gz'),sep='\t',index_col=0)
    df_bin['position']=df_bin.index.to_series().apply(lambda x:x.split('_')[0]+':'+str(int(x.split('_')[1])*100000)+'-'+str((int(x.split('_')[1])+1)*100000))
    cols=comp.columns.tolist()
    df_bin['GroupID']=df_bin.index.to_series().map(comp.apply(lambda x:cols[np.argmax(x)], axis=1).to_dict())
    # sig_bins = (df_bin['padj']<1e-5)
    sig_bins = zscore(df_bin['sample_maha'])>norm.isf(0.025) #Inverse survival function, norm.isf(0.025) = 1.95,norm.sf(1.95)=0.025, area of > 1.95, right part in norm curve
    print(sig_bins.sum()) #selb is a series, index are bins ID, values are True/False

    df1=df_bin.loc[sig_bins]
    fig, ax = plt.subplots(figsize=(2,2), dpi=300)
    if 'ATAC_corr' in df_bin.columns.tolist():
        plot_data=[
            df1['mCG_corr'],
            df1['mCH_corr'],
            df1['ATAC_corr'],
                  ]
        xticklabels=['mCG', 'mCH', 'ATAC']
        colors=['limegreen','royalblue','darkorange']
    else:
        plot_data=[
            df1['mCG_corr'],
            df1['mCH_corr']
        ]
        xticklabels=['mCG', 'mCH']
        colors=['limegreen','royalblue']
    ax.grid(axis='y',which='major',zorder=-1,linestyle='--')
    sns.violinplot(plot_data, ax=ax,palette=colors)
    ax.set_xticklabels(xticklabels)
    ax.set_title(group,fontsize=10)
    ax.set_ylabel('Correlation with Comp Score',fontsize=8)
    plt.savefig(os.path.join(figdir,f'{group}_multiomics_correlation_violinplot.pdf'), transparent=True,bbox_inches='tight')
    plt.show()
    return df_bin,comp,sig_bins

def plot_heatmap(celltypes,frac_dict,atac_data,comp,use_split_order,stratify_col,title):
    from PyComplexHeatmap.utils import despine
    mcg=frac_dict['CGN']
    use_cols=list(set(mcg.columns.tolist()) & set(celltypes))
    mcg = mcg.loc[df_bin.index.tolist(), use_cols]
    mch=frac_dict['CHN']
    mch = mch.loc[df_bin.index.tolist(), use_cols]
    group_name=group_col if stratify_col is None else 'GroupID'
    df_cols=comp.columns.to_frame(name=group_name)
    if not stratify_col is None:
        if stratify_figdir is None:
            df_cols[stratify_col]=df_cols[group_name].apply(lambda x:x.split('+')[0])
            df_cols[group_col]=df_cols[group_name].apply(lambda x:x.split('+')[1])
        else:
            df_cols[group_col]=df_cols[group_name]
    if not parent_col is None:
        df_cols[parent_col]=df_cols[group_name].map(group2parent) #group2parent is a dict: GroupID -> parent (stratify) or cell type -> parent
        df_cols.sort_values([parent_col,group_name],inplace=True)
        if not use_split_order is None:
            use_split_order=[o for o in use_split_order if o in df_cols[parent_col].tolist()]
    else:
        if use_split_order is None:
            df_cols.sort_values(group_name,inplace=True)
        else:
            use_split_order=[o for o in use_split_order if o in df_cols.index.tolist()]
            df_cols=df_cols.loc[use_split_order]
    keep_rows=list(set(df_cols.index.tolist()) & set(mcg.columns.tolist()))
    df_cols=df_cols.loc[keep_rows]

    df_rows=df_bin.loc[sig_bins,['chr','start','end','sample_maha','comp_std','GroupID']]
    if not parent_col is None:
        df_rows[parent_col]=df_rows.GroupID.map(group2parent)

    row_order=[]
    for g in df_cols[group_name].tolist():
        rows=df_rows.loc[df_rows.GroupID==g].sort_values('sample_maha',ascending=False).index.tolist()
        row_order.extend(rows)
    df_rows=df_rows.loc[row_order]

    tmpcg = mcg.loc[df_rows.index,df_cols[group_name].tolist()]
    tmpch = mch.loc[df_rows.index,df_cols[group_name].tolist()]
    if not atac_data is None:
        atac=atac_data.loc[df_rows.index,df_cols[group_name].tolist()]
    tmp3c = comp.loc[df_rows.index,df_cols[group_name].tolist()]
    print(tmp3c.shape,tmpcg.shape,tmpch.shape)

    if not parent_col is None:
        colors={}
        for parent in parent2group:
            if parent not in df_rows[parent_col].tolist():
                continue
            colors[parent]=parent_colors[parent]

        row_ha_dict={
                    parent_col:anno_simple(df_rows[parent_col],colors=colors,legend=False),
                    }
        row_ha = HeatmapAnnotation(**row_ha_dict,axis=0,
                 verbose=0,label_kws=dict(visible=False))
        row_split=df_rows[parent_col]
        row_split_order=df_cols[parent_col].unique().tolist() if use_split_order is None else use_split_order

        col_ha_dict={
                    parent_col:anno_simple(
                        df_cols[parent_col],#height=5,
                         colors=colors,
                         add_text=False,text_kws={'fontweight':'bold','color':'white'},legend=True),
                    }
        col_ha = HeatmapAnnotation(**col_ha_dict,axis=1,
                 verbose=0,label_kws=dict(visible=False))
        col_split=df_cols[parent_col]
        col_split_order=df_cols[parent_col].unique().tolist() if use_split_order is None else use_split_order
    else:
        col_ha=None
        col_split=None
        col_split_order=None
        row_ha=None
        row_split=None
        row_split_order=None
    print(df_rows.head())
    print(df_cols.head())
    print(col_split_order,row_split_order)

    cm1 = ClusterMapPlotter(
                data=tmp3c, top_annotation=col_ha, left_annotation=row_ha,
                show_rownames=False, show_colnames=True,col_cluster=False,row_cluster=False,
                row_dendrogram=False, col_dendrogram=False,vmin=-3,vmax=3,
                row_cluster_method='ward',row_cluster_metric='cosine',
                col_split=col_split, col_split_order=col_split_order,
                xticklabels_kws={'labelrotation':-60,'labelcolor':'black','labelsize':10},xticklabels=True,
                col_split_gap=0.4,cmap='BlueDarkRed18',#turbo, parula, viridis,RedYellowBlue_r,BlueOrangeRed,
                row_split=row_split,row_split_gap=0.8,row_split_order=row_split_order,
                rasterized=True, #row_split_gap=0.1,vmax=1,vmin=0,center=0.5,
                plot=False,label='comp_score',legend_kws=dict(extend='both',extendfrac=0.1),
                # z_score=0,#standard_scale=0,
                verbose=1)

    cm2 = ClusterMapPlotter(
                data=tmpcg, top_annotation=col_ha, #left_annotation=row_ha,
                show_rownames=False, show_colnames=True,col_cluster=False,row_cluster=False,
                row_dendrogram=False, col_dendrogram=False,xticklabels=True,
                col_split=col_split, col_split_order=col_split_order,
                xticklabels_kws={'labelrotation':-60,'labelcolor':'black','labelsize':10},
                col_split_gap=0.4,row_split_gap=0.8,rasterized=True,
                plot=False,cmap='parula',label='mCG',
                z_score=0,
                legend_kws=dict(extend='both',extendfrac=0.1),
                verbose=0)

    cm3 = ClusterMapPlotter(
                data=tmpch, top_annotation=col_ha, #left_annotation=row_ha,
                show_rownames=False, show_colnames=True,col_cluster=False,row_cluster=False,
                row_dendrogram=False, col_dendrogram=False,xticklabels=True,
                col_split=col_split, col_split_order=col_split_order,
                xticklabels_kws={'labelrotation':-60,'labelcolor':'black','labelsize':10},
                col_split_gap=0.4,row_split_gap=0.8,rasterized=True,
                plot=False,cmap='agGrnYl',label='mCH',
                z_score=0,
                legend_kws=dict(extend='both',extendfrac=0.1),# standard_scale=1,
                verbose=0) #or Greys,BlueOrange12,
    if not atac_data is None:
        cm4 = ClusterMapPlotter(
                data=atac, top_annotation=col_ha, #left_annotation=row_ha,
                show_rownames=False, show_colnames=True,col_cluster=False,row_cluster=False,
                row_dendrogram=False, col_dendrogram=False,xticklabels=True,
                col_split=col_split, col_split_order=col_split_order,
                xticklabels_kws={'labelrotation':-60,'labelcolor':'black','labelsize':10},
                col_split_gap=0.4,row_split_gap=0.8,rasterized=True,
                plot=False,cmap='BlueDarkOrange12',label='ATAC\nlog(CPM)',
                z_score=0,
                legend_kws=dict(extend='both',extendfrac=0.1),# standard_scale=1,
                verbose=0)
        cmlist=[cm1,cm4,cm2,cm3]
    else:
        cmlist=[cm1,cm2,cm3]

    nrows,ncols=tmp3c.shape
    width=max(ncols*0.6,5) if atac_data is None else max(ncols*0.8,5)
    plt.figure(figsize=(width,10))
    ax,legend_axes=composite(cmlist=cmlist, main=0,legend_hpad=2,col_gap=5,legend_gap=10)
    cm1.ax.set_title("Compartment score",fontsize=14)
    cm2.ax.set_title("mCG",fontsize=14)
    cm3.ax.set_title("mCH",fontsize=14)
    if not atac_data is None:
        cm4.ax.set_title("ATAC",fontsize=14)
    ax.set_ylabel(str(tmp3c.shape[0])+" Diff Comp",color='black',fontsize=14)
    # ax.yaxis.set_label_position('left')
    ax.set_axis_on()
    despine(ax=ax, left=True, bottom=True,top=True,right=True)
    ax.tick_params(axis="both",which="both",left=False,
                    right=False,labelleft=False,labelright=False,
                    top=False,bottom=False,labeltop=False,labelbottom=False)
    # ax.set_title("Title",y=1.03,fontdict={'fontweight':'bold','color':'red'})
    plt.savefig(os.path.join(figdir,f'{title}_diffcomp_multiple_omics_heatmap.pdf'), transparent=True,bbox_inches='tight')
    plt.show()

    # plot 3C and ATAC only
    if not atac_data is None:
        width=max(ncols*0.4,5)
        plt.figure(figsize=(width,10))
        ax,legend_axes=composite(cmlist=[cm1,cm4], main=0,legend_hpad=2,col_gap=5,legend_gap=10)
        cm1.ax.set_title("Compartment score",fontsize=14)
        cm4.ax.set_title("ATAC",fontsize=14)
        ax.set_ylabel(str(tmp3c.shape[0])+" Diff Comp",color='black',fontsize=14)
        # ax.yaxis.set_label_position('left')
        ax.set_axis_on()
        despine(ax=ax, left=True, bottom=True,top=True,right=True)
        ax.tick_params(axis="both",which="both",left=False,
                        right=False,labelleft=False,labelright=False,
                        top=False,bottom=False,labeltop=False,labelbottom=False)
        plt.savefig(os.path.join(figdir,f'{title}_diffcomp_atac_heatmap.pdf'), transparent=True,bbox_inches='tight')
        plt.show()
    return df_rows

In [ ]:
if stratify_col is None:
    parent='merged'
    use_dict=parent2group
    r=get_comp_data(parent,use_dict)
    if not r==False:
        df_bin,comp,sig_bins=r
        celltypes=use_dict[parent]
        df_rows=plot_heatmap(celltypes,frac_dict,atac_data,comp,split_order,stratify_col,group_col)
        # write diff comp to bed (used for GREAT enrichment analysis)
        outdir1=os.path.join(figdir,"GREAT-diffcomp")
        if not os.path.exists(outdir1):
            os.makedirs(outdir1,exist_ok=True)
        for group in df_rows.GroupID.unique():
            df=df_rows.loc[df_rows.GroupID==group,['chr','start','end']]
            df.sort_values(['chr','start'],ascending=True,inplace=True)
            df.rename(columns={'chr':'#chr'},inplace=True)
            df.to_csv(os.path.join(outdir1,f"{group}.bed"),sep='\t',index=False)
else:
    for stratify in stratify2group:
        r=get_comp_data(stratify,stratify2group)
        if r==False:
            print("diff compartment input file not existed")
            continue
        df_bin,comp,sig_bins=r
        celltypes=stratify2group[stratify]
        try:
            plot_heatmap(celltypes,frac_dict,atac_data,comp,split_order,stratify_col,stratify)
        except:
            continue

comp_score are quantile normalized score, Quantile normalization ensures that PC1 values across datasets have the same distribution, making them directly comparable.

In [ ]:
# # calculate pearson correlation for each cell type across all bins
# infile=os.path.join(
#     diff_comp_dir,
#     'merged/DifferentialResult/merged/fdr_result/differential.intra_sample_combined.pcQnm.bedGraph')
# df_comp = pd.read_csv(infile,sep='\t')
# df_comp.index = df_comp['chr'] + '_' + (df_comp['start'] // res).astype(str)
# df_comp.columns=[col.replace('_AND_','-').replace('_',' ').replace('_PLUS_','+') if col!="sample_maha" else col for col in df_comp.columns.tolist()]
# df_comp=df_comp.loc[~ df_comp['chr'].isin(['chrX','chrM','chrY'])]
# usecols=list(set(group_order) & set(df_comp.columns.tolist()))
# comp = df_comp[usecols]

# mcg=frac_dict['CGN']
# use_cols=list(set(mcg.columns.tolist()) & set(usecols))
# mcg = mcg.loc[df_comp.index.tolist(),use_cols] #methylation fraction for each 100kb bin in each cell type

# mch=frac_dict['CHN']
# mch = mch.loc[df_comp.index.tolist(),use_cols]

# comp=comp.loc[:,use_cols]
# if not pseudobulk_atac_adata_path is None:
#     atac = atac_data.loc[df_comp.index.tolist(),use_cols]
# R=[]
# for ct in use_cols:
#     r_cg=pearsonr(mcg[ct].values, comp[ct].values)
#     r_ch=pearsonr(mch[ct].values, comp[ct].values)
#     r_atac=pearsonr(atac[ct].values, comp[ct].values)
#     R.append([ct,r_cg[0],r_cg[1],r_ch[0],r_ch[1],r_atac[0],r_atac[1]])
# data=pd.DataFrame(R,columns=['CellType','CG_corr','CG_pvalue','CH_corr','CH_pvalue','ATAC_corr','ATAC_pvalue']).set_index('CellType')
# data.to_csv(os.path.join(figdir,f'{group_col}.multiomics_correlation_across_bins.tsv'),sep='\t')

# df1=data.loc[:,['CG_corr','CH_corr','ATAC_corr']]
# df1.columns=[col.replace('_corr','') for col in df1.columns.tolist()]

# df_rows=pd.Series(group2parent)
# df_rows=df_rows.loc[df1.index.tolist()]
# colors={}
# for parent in parent2group:
#     if parent not in df_rows.tolist():
#         continue
#     colors[parent]=parent_colors[parent]
# row_ha_dict={
#     parent_col:anno_simple(df_rows,colors=colors,legend=False),
# }
# row_ha = HeatmapAnnotation(**row_ha_dict,axis=0,
#          verbose=0,label_side='bottom',label_kws=dict(visible=False))

# plt.figure(figsize=(3,5))
# cm = ClusterMapPlotter(
#                 data=df1, left_annotation=row_ha,
#                 show_rownames=True, show_colnames=True,col_cluster=False,row_cluster=True,
#                 row_dendrogram=False, col_dendrogram=False,
#                 row_split=df_rows,xticklabels=True,yticklabels=True,
#                 xticklabels_kws={'labelrotation':-30,'labelcolor':'black','labelsize':10,'bottom':True},
#                 yticklabels_kws=dict(right=True),
#                 row_split_gap=0.5,rasterized=True,#linecolor='white',linewidth=0.05,
#                 cmap='Spectral_r',label='Correlation',annot=True,
#                 legend_kws=dict(extend='both',extendfrac=0.1),# standard_scale=1,
#                 verbose=0)
# plt.savefig(os.path.join(figdir,f'{group_col}.multiomics_correlation_across_bins.pdf'), 
#             transparent=True,bbox_inches='tight',dpi=300)
# plt.show()

In [ ]:
if stratify_col is None:
    order=df_bin.loc[sig_bins]['chr'].value_counts().index.tolist()
    plt.figure(figsize=(6,3.5))
    sns.countplot(df_bin.loc[sig_bins],x='chr',order=order,palette='Spectral')
    plt.setp(plt.gca().get_xticklabels(), rotation=-45, ha='left')
    plt.savefig(os.path.join(figdir,f'No.of_diff_comp_per_chr_countplot.pdf'), transparent=True,bbox_inches='tight')
    plt.show()

df_bin.loc[sig_bins].sort_values('sample_maha',ascending=False)

In [ ]:
# Top 3 diff comp for each cell type
df_bin.groupby('GroupID').apply(lambda x:x.sort_values('sample_maha',ascending=False).position.head(3).tolist()).to_dict()

In [ ]:
parent='merged'
use_dict=parent2group
r=get_comp_data(parent,use_dict)
df_bin,comp,sig_bins=r

In [ ]:
# write compartment scores (can be compared across cell types) to table
group='merged'
res=100000
group_name=group.replace(' ','_')
diff_comp_dir=os.path.expanduser("~/Projects/BG/HIC/BG_Subclass/compartment/SubclassDiffComp")
infile=os.path.join(diff_comp_dir,
                f'{group_name}/DifferentialResult/{group_name}/fdr_result/differential.intra_sample_combined.pcQnm.bedGraph')
df_comp = pd.read_csv(infile, sep='\t')
df_comp.index = df_comp['chr'] + '_' + (df_comp['start'] // res).astype(str)
df_comp.columns=[col.replace('_AND_','-').replace('_',' ').replace('_PLUS_','+') for col in df_comp.columns.tolist()]
df_comp=df_comp.loc[~ df_comp['chr'].isin(['chrX','chrM','chrY'])]
usecols=[col for col in df_comp.columns.tolist() if col in group_colors]
df_comp=df_comp.loc[:,['chr','start','end']+usecols]
df_comp.to_csv(os.path.join(figdir,'NormalizedCompartmentScores.tsv'),sep='\t')
df_comp

In [ ]:
parent='merged'
use_dict=parent2group
r=get_comp_data(parent,use_dict)
df_bin,comp,sig_bins=r
df_rows=df_bin.loc[sig_bins,['chr','start','end','sample_maha','comp_std','GroupID']]
df_rows.to_csv(os.path.join(figdir,'DiffCompartment.tsv'),sep='\t')